In [3]:
import numpy as np

from ncs.compressed_sensing_module import measure_and_reconstruct
from ncs.wavelet_module import inverse_transform
%load_ext autoreload
%autoreload 2

In [7]:
from ncs.sparse_signal_generator import generate_tree_sparse_coeffs

In [9]:
n_power = 10  # signal length 1024
n = 2 ** n_power
tree_sparsity = 10  # 1%
wavelet = 'haar'

signal_count = 10
reconstruction_attempts = 10

In [10]:
sparse_coeffs = generate_tree_sparse_coeffs(
    power=n_power,
    count=signal_count,
    tree_sparsity=tree_sparsity,
    wavelet=wavelet,
)

In [ ]:
m = n / 2
measurement_mode = "gaussian"
reconstruction_mode = "CoSaMP"

mse_values = []
missed_support_lengths = []

for sparse_coeff in sparse_coeffs:
    for _ in range(reconstruction_attempts):
        x_hat = measure_and_reconstruct(
            measurement_mode=measurement_mode,
            m=m,
            reconstruction_mode=reconstruction_mode,
            coeffs_x=sparse_coeff,
            target_tree_sparsity=tree_sparsity,
        )

        sparse_z = inverse_transform(sparse_coeff)
        reconstructed_z = inverse_transform(x_hat)

        missed_support_len = len(sparse_coeff.support - x_hat.support)
        mse = np.mean((sparse_z - reconstructed_z) ** 2)

        mse_values.append(mse)
        missed_support_lengths.append(missed_support_len)

avg_mse = np.mean(mse_values)
avg_missed_support = np.mean(missed_support_lengths)